<a href="https://colab.research.google.com/github/RomanLuja/ML/blob/main/project_kickoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ппроєкт: від Kaggle-датасету до першої моделі

## 1. Проєкт: що і як

**Завдання:** побудувати прогнозну модель на реальному датасеті з Kaggle: очистити дані, створити
ознаки, порівняти моделі з коректною валідацією, пояснити модель і написати висновок для бізнесу.

**Датасети на вибір** (усі публічні, завантажуються через `kagglehub` без реєстрації):

| № | Датасет | Задача | Чим цікавий |
|---|---|---|---|
| 1 | [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) | класифікація: чи піде клієнт | числа, записані як текст; бізнес-розрахунок утримання |
| 2 | [Hotel Booking Demand](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand) | класифікація: чи скасують бронювання | **витоки цілі**, дублікати, валідація за часом, 119 тис. рядків |
| 3 | [New York City Airbnb](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data) | регресія: ціна за ніч | викиди, пропуски, багато категорій, логарифм цілі |

Складність зростає від 1 до 3

---
## Практика: старт власного проєкту

1. Відкрийте `project_template.ipynb`, збережіть копію у свій Google Drive
   (`Файл → Зберегти копію на Диску`).
2. Оберіть датасет і змініть `DATASET` і `FILE` у комірці завантаження.
3. Заповніть **розділ 1** (постановка). Покажіть викладачу до кінця заняття.
4. Почніть **розділ 2** (аудит): типи колонок, пропуски, дублікати, ціль, витоки.

**Підказки до датасетів (без спойлерів):**
- **Telco:** перевірте тип кожної «числової» колонки — чи справді вона число?
- **Hotel:** для кожної колонки запитайте: «Чи відомо це в момент бронювання?» Подумайте, як
  розділити дані, щоб модель прогнозувала майбутні бронювання.
- **Airbnb:** подивіться на мінімальні й максимальні ціни. Чи всі оголошення реальні?

## 2. Демонстрація: проєкт у мініатюрі

Пройдемо всі розділи шаблону швидко на датасеті [IBM HR Analytics Employee Attrition](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)
(його немає серед варіантів проєкту). Мета — побачити **порядок дій**, а не ідеальний результат.

### 0. Завантаження з Kaggle

| Columns | Values |
| ------- | ------ |
| Age | Вік працівника |


In [ ]:
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

path = kagglehub.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")
print(os.listdir(path))
ibm = pd.read_csv(os.path.join(path, "WA_Fn-UseC_-HR-Employee-Attrition.csv"))
print(ibm.shape)
ibm.head(3)

Using Colab cache for faster access to the 'ibm-hr-analytics-attrition-dataset' dataset.
['WA_Fn-UseC_-HR-Employee-Attrition.csv']
(1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0


### 1. Постановка

- **Замовник:** HR; рішення — кому запропонувати програму утримання.
- **Ціль:** `Attrition` (Yes/No) → класифікація; один рядок — працівник.
- **Метрика:** ROC-AUC (ранжування за ризиком) + recall у топ-15 % списку; бізнес — вартість заміни.

### 2. Аудит

In [ ]:
ibm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [ ]:
ibm['Attrition'].value_counts()

,count
Attrition,
No,1233
Yes,237


In [ ]:
print("Дублікати:", ibm.duplicated().sum())
print("Пропуски:", ibm.isna().sum().sum())
print("Частка звільнень:", (ibm['Attrition'] == 'Yes').mean().round(3))

Дублікати: 0
Пропуски: 0
Частка звільнень: 0.161


In [ ]:
print('Ідентифікатор: EmployeeNumber', ibm['EmployeeNumber'].is_unique)

Ідентифікатор: EmployeeNumber True


In [ ]:
constant = [c for c in ibm.columns if ibm[c].nunique() == 1]
print('Константи:', constant)

Константи: ['EmployeeCount', 'Over18', 'StandardHours']


In [ ]:
df = ibm.drop(columns=constant + ['EmployeeNumber'])
df['Attrition'] = (df['Attrition'] == 'Yes').astype(int)

In [ ]:
df.head(3)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0


### 3. Ознаки

In [ ]:
df['IncomePerLevel'] = df['MonthlyIncome'] / df['JobLevel']
df['YearsWithoutPromotionShare'] = df['YearsSinceLastPromotion'] / (df['YearsAtCompany'] + 1)
df['CompaniesPerYear'] = df['NumCompaniesWorked'] / (df['TotalWorkingYears'] + 1)

### 4. Моделювання і валідація

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

X = df.drop(columns='Attrition')
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def prep(scale):
    return ColumnTransformer([
        ('num', StandardScaler() if scale else 'passthrough', make_column_selector(dtype_include='number')),
        ('cat', OneHotEncoder(handle_unknown='ignore'), make_column_selector(dtype_exclude='number'))
    ])

candidates = {
    "Орієнтир": Pipeline([
                    ('prep', prep(False)),
                    ('clf', DummyClassifier(strategy='prior'))
                ]),
    "Логістична регресія": Pipeline([
                    ('prep', prep(True)),
                    ('clf', LogisticRegression(max_iter=2000))
                ]),
}



### 5. Інтерпретація

### 6. Висновок